# Phase 1 — Baseline OCT Segmentation Benchmarking
Run this notebook top-to-bottom in Google Colab.

**What this notebook does:**
1. Mounts Google Drive and clones / sets up the project.
2. Installs all dependencies.
3. Loads the trained UNet++ EfficientNet-B4 checkpoint.
4. Runs full baseline benchmarking:
   - Dice, IoU, Pixel Accuracy, Sensitivity, Specificity
   - GFLOPs, Parameters, Memory
   - FPS, Latency
   - Energy (codecarbon fallback for Colab)
5. Exports all metrics to CSV.
6. Saves qualitative visualisations.


In [ ]:
# ── 1. Mount Drive ──────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── 2. Copy project to /content (fast local SSD) ────────────────────────────
import shutil, os

PROJECT_DRIVE = '/content/drive/MyDrive/FYP/oct_seg_phase1'
PROJECT_LOCAL = '/content/oct_seg_phase1'

if os.path.exists(PROJECT_LOCAL):
    shutil.rmtree(PROJECT_LOCAL)
shutil.copytree(PROJECT_DRIVE, PROJECT_LOCAL)
os.chdir(PROJECT_LOCAL)
print(f'Working dir: {os.getcwd()}')

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
# (skip packages already present by checking importability)
!python install_dependencies.py

In [ ]:
# ── 4. Add project root to PYTHONPATH ───────────────────────────────────────
import sys
sys.path.insert(0, '/content/oct_seg_phase1')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__} | Device: {device}')

In [ ]:
# ── 5. Load config ───────────────────────────────────────────────────────────
from utils.config_loader import load_config, get_class_info

cfg        = load_config('configs/config.yaml')
class_info = get_class_info(cfg)
class_names = [vars(cfg.classes.names)[str(i)] for i in range(cfg.classes.num_classes)]

print('Config loaded.')
print(f'Classes: {class_names}')
print(f'Device:  {cfg.inference.device}')

In [ ]:
# ── 6. Run full evaluation ───────────────────────────────────────────────────
# This single call executes all 9 evaluation steps and exports the CSV.
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

from evaluation.evaluate_baseline import run_evaluation

bundle = run_evaluation(cfg, experiment_name='baseline_full_image')

In [ ]:
# ── 7. Display saved metrics CSV ─────────────────────────────────────────────
import pandas as pd
df = pd.read_csv(cfg.evaluation.metrics_output_csv)
df.T   # Transpose for easier reading in notebook

In [ ]:
# ── 8. Show visualisations inline ────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

viz_dir = cfg.evaluation.visualizations_dir
figs    = sorted([f for f in os.listdir(viz_dir) if f.endswith('.png')])

for fname in figs:
    img = mpimg.imread(os.path.join(viz_dir, fname))
    plt.figure(figsize=(14, 6))
    plt.imshow(img)
    plt.title(fname, fontsize=10)
    plt.axis('off')
    plt.show()

In [ ]:
# ── 9. Optional: ad-hoc single-image inference demo ─────────────────────────
from models.model_loader import get_model
from inference.baseline_inference import predict_single
import matplotlib.pyplot as plt

model = get_model(cfg)

# Replace with any OCT image path
DEMO_IMAGE = cfg.data.image_dir + '/' + os.listdir(cfg.data.image_dir)[0]

pred_cls, pred_rgb = predict_single(
    model, DEMO_IMAGE,
    image_size = cfg.data.image_size,
    device     = cfg.inference.device,
)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
import cv2
img = cv2.imread(DEMO_IMAGE, cv2.IMREAD_GRAYSCALE)
axes[0].imshow(img,      cmap='gray'); axes[0].set_title('OCT Input')
axes[1].imshow(pred_rgb);              axes[1].set_title('Predicted Mask')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## Results stored at
- `results/metrics/baseline_metrics.csv` — all benchmark numbers
- `results/visualizations/` — qualitative figures
- `results/predictions/` — saved prediction PNGs
- `results/run.log` — full execution log

## Next: Phase 2
Phase 2 will add YOLOv8n ROI detection before segmentation.
The `MetricsBundle` CSV schema is designed to accept multiple rows,
so baseline vs ROI-guided results can be compared directly.
